In [1]:


import xml.etree.ElementTree as ET
import asyncio
from neo4j import AsyncGraphDatabase
from sentence_transformers import SentenceTransformer, util
import xml.etree.ElementTree as ET
from neo4j import GraphDatabase


# === CONFIG ===
NEO4J_URI = "neo4j+s://ddf3e3b8.databases.neo4j.io"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "c5bFc-_D_rviELGy0J7I_LC2Ltbhgr1eJ-_fOODWOiA"
NEO4J_DB = "neo4j"
XML_FILE = r"D:\GraphRAG_Project\omdxe11337.xml"
RELEVANT_TAGS = {"component", "procedure", "step"}
BATCH_SIZE = 200

# === GLOBAL ===
nodes = []
relationships = []
model = SentenceTransformer('all-MiniLM-L6-v2')


# === CONNECT ===
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# === Helpers ===
def clean_tag(tag):
    return tag.split("}")[-1]

def extract_text(elem):
    texts = [elem.text.strip()] if elem.text and elem.text.strip() else []
    for child in elem:
        texts.append(extract_text(child))
        if child.tail and child.tail.strip():
            texts.append(child.tail.strip())
    return " ".join(texts).strip()

def process_element(elem, parent=None):
    tag = clean_tag(elem.tag)
    if tag in RELEVANT_TAGS:
        node_id = elem.attrib.get("id", f"{tag}_{hash(elem)}")
        content = extract_text(elem)
        props = {"id": node_id, "tag": tag, "content": content, **elem.attrib}
        nodes.append({"id": node_id, "tag": tag, "props": props})
        if parent:
            relationships.append({
                "from_id": parent["id"],
                "to_id": node_id,
                "from_label": parent["tag"],
                "to_label": tag,
                "type": "CONTAINS"
            })
        parent = {"id": node_id, "tag": tag}
    for child in elem:
        process_element(child, parent)

async def write_batch(tx, node_batch, rel_batch):
    for node in node_batch:
        await tx.run("""
            MERGE (n:{tag} {id: $id})
            SET n += $props
        """.replace("{tag}", node['tag']), id=node["id"], props=node["props"])

    for rel in rel_batch:
        await tx.run(f"""
            MATCH (a:{rel['from_label']} {{id: $from_id}}), (b:{rel['to_label']} {{id: $to_id}})
            MERGE (a)-[:{rel['type']}]->(b)
        """, from_id=rel["from_id"], to_id=rel["to_id"])

async def load_to_neo4j():
    driver = AsyncGraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    async with driver.session(database=NEO4J_DB) as session:
        for i in range(0, len(nodes), BATCH_SIZE):
            node_batch = nodes[i:i + BATCH_SIZE]
            rel_batch = relationships[i:i + BATCH_SIZE]
            await session.execute_write(write_batch, node_batch, rel_batch)
    await driver.close()

async def clear_database():
    driver = AsyncGraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    async with driver.session(database=NEO4J_DB) as session:
        await session.run("MATCH (n) DETACH DELETE n")
    await driver.close()

async def vector_query_graph_async(user_question, limit=5):
    driver = AsyncGraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    async with driver.session(database=NEO4J_DB) as session:
        result = await session.run("MATCH (n) WHERE n.content IS NOT NULL RETURN n.id AS id, n.tag AS tag, n.content AS content")
        candidates = [record async for record in result]
    await driver.close()

    question_emb = model.encode(user_question, convert_to_tensor=True)
    scores = []
    for c in candidates:
        content_emb = model.encode(c["content"], convert_to_tensor=True)
        sim = util.pytorch_cos_sim(question_emb, content_emb).item()
        scores.append((sim, c))

    top_results = sorted(scores, key=lambda x: x[0], reverse=True)[:limit]
    return [r[1] for r in top_results]

def vector_query_graph(user_question, limit=5):
    # 1. Load candidate nodes
    with driver.session(database=NEO4J_DB) as session:
        result = session.run("MATCH (n) WHERE n.content IS NOT NULL RETURN n.id AS id, n.tag AS tag, n.content AS content")
        candidates = [r.data() for r in result]

    # 2. Compute embedding similarity
    question_emb = model.encode(user_question, convert_to_tensor=True)
    scores = []
    for c in candidates:
        content_emb = model.encode(c['content'], convert_to_tensor=True)
        sim = util.pytorch_cos_sim(question_emb, content_emb).item()
        scores.append((sim, c))

    # 3. Sort by similarity and return top-K
    top_results = sorted(scores, key=lambda x: x[0], reverse=True)[:limit]
    return [r[1] for r in top_results]
    
async def main():
    await clear_database()
    print(" Parsing XML...")
    tree = ET.parse(XML_FILE)
    process_element(tree.getroot())
    print(f"Parsed {len(nodes)} nodes and {len(relationships)} relationships.")
    print("Loading to Neo4j...")
    await load_to_neo4j()
    print("Done!")

In [ ]:
import nest_asyncio
import asyncio

nest_asyncio.apply()
await main()  # 

In [2]:
user_question = "How to prevent starter damage?"
vector_query_graph(user_question, limit=3)

[{'id': 'step_144972959019',
  'tag': 'step',
  'content': 'Gas given off by batteries is explosive. Avoid sparks\nnear batteries. Never connect jumper cables with the key switch or\nbattery disconnect switch ON. Never jump-start with more than 12 V. Remove protective caps from posts.'},
 {'id': 'step_144972955553',
  'tag': 'step',
  'content': 'BATTERIES ARE NEGATIVE GROUNDED ONLY. Always connect\nthe battery ground strap to the negative (-) posts of the battery.\nConnect the starter cable to the positive (+) post of the battery.\nReversed polarity in the battery or alternator connections results\nin permanent damage to the electrical system. Connect the ground strap\nto the negative (-) terminal last. Batteries must have same terminal locations. Turn off all of the switches and accessories. Clean\nthe battery posts and the terminals.'},
 {'id': 'step_144964961140',
  'tag': 'step',
  'content': 'Key Switch Sound horn before starting engine to warn others to stay\nclear from machine.

In [4]:
user_question = "How do I replace headlight assembly?"
results = vector_query_graph(user_question, limit=3)
results

[{'id': 'step_144972975973',
  'tag': 'step',
  'content': 'Remove and replace the\nheadlight assembly.'},
 {'id': 'step_144972975863',
  'tag': 'step',
  'content': 'Headlight Connector  Headlight Assembly A Connector B Nut C Cap Screw Raise the feeder house, engage the feeder house safety lock,\nshut OFF engine, set park brake, and remove key before replacing or\nadjusting the headlights. Disconnect the wiring harness connector (A) from the\nheadlight assembly.'},
 {'id': 'step_144972981073',
  'tag': 'step',
  'content': 'Remove and replace the\nlight assembly.'}]

In [6]:
def build_enriched_prompt(context: str, user_question: str) -> str:
    return f"""
You are a knowledgeable and helpful technical assistant.

Below are relevant excerpts from a service manual. Use this information to answer the user’s question with detailed, full-sentence instructions. Include safety warnings, tool usage, and logical reasoning if necessary. If the answer is a process, explain the steps in complete, well-structured sentences.

--- Service Manual Excerpts ---
{context}
--------------------------------

User Question: {user_question}

Please provide a thorough and helpful answer using full sentences:
"""


In [7]:
from transformers import pipeline
model_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-large",  # or flan-ul2
    device_map="auto",
    torch_dtype="auto"
)

def generate_answer_from_context(results, user_question):
    if not results:
        return "⚠️ No relevant content found to answer the question."

    context = "\n".join([
        f"{r.get('tag', 'step')} ({r.get('id', 'unknown')}): {r.get('content', '').strip()}"
        for r in results if r.get('content')
    ])

    prompt = build_enriched_prompt(context, user_question)

    response = model_pipeline(
        prompt,
        max_new_tokens=512,
        temperature=0.9,           # 🎨 slight creativity
        top_k=50,                  # pick from top 50 tokens
        top_p=0.95,                 # allow 90% cumulative prob
        repetition_penalty=1.2     # 🛡️ avoid repeating phrases
    )

    return response[0]["generated_text"].strip().split("Answer:")[-1].strip()


Some parameters are on the meta device because they were offloaded to the cpu.
Device set to use cuda:0


In [34]:
user_question = "How to prevent starter damage?"
results = vector_query_graph(user_question, limit=3)
generate_answer_from_context(results =results,user_question =  user_question)

'To prevent starter damage, do not operate starter for more than 30 seconds at a time. If engine does not start, wait at least 2 minutes before trying again.'

In [35]:
results

[{'id': 'step_144972959019',
  'tag': 'step',
  'content': 'Gas given off by batteries is explosive. Avoid sparks\nnear batteries. Never connect jumper cables with the key switch or\nbattery disconnect switch ON. Never jump-start with more than 12 V. Remove protective caps from posts.'},
 {'id': 'step_144972955553',
  'tag': 'step',
  'content': 'BATTERIES ARE NEGATIVE GROUNDED ONLY. Always connect\nthe battery ground strap to the negative (-) posts of the battery.\nConnect the starter cable to the positive (+) post of the battery.\nReversed polarity in the battery or alternator connections results\nin permanent damage to the electrical system. Connect the ground strap\nto the negative (-) terminal last. Batteries must have same terminal locations. Turn off all of the switches and accessories. Clean\nthe battery posts and the terminals.'},
 {'id': 'step_144964961140',
  'tag': 'step',
  'content': 'Key Switch Sound horn before starting engine to warn others to stay\nclear from machine.